# Snake AI with Q-Learning

This notebook builds a tiny snake environment and trains a Q-learning agent.


In [ ]:
import random
from collections import deque, defaultdict

import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)


## 1. Environment


In [ ]:
DIRECTIONS = [
    (1, 0),   # right
    (0, 1),   # down
    (-1, 0),  # left
    (0, -1),  # up
]

class SnakeEnv:
    def __init__(self, size=8, max_steps=100):
        self.size = size
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        center = self.size // 2
        self.direction = random.randint(0, 3)
        dx, dy = DIRECTIONS[self.direction]
        head = (center, center)
        body = (center - dx, center - dy)
        self.snake = deque([head, body])
        self.score = 0
        self.steps = 0
        self._place_food()
        return self.get_state()

    def _place_food(self):
        while True:
            food = (random.randint(0, self.size - 1), random.randint(0, self.size - 1))
            if food not in self.snake:
                self.food = food
                return

    def _is_collision(self, point):
        x, y = point
        if x < 0 or x >= self.size or y < 0 or y >= self.size:
            return True
        if point in list(self.snake):
            return True
        return False

    def _next_head(self, direction_idx):
        dx, dy = DIRECTIONS[direction_idx]
        head_x, head_y = self.snake[0]
        return (head_x + dx, head_y + dy)

    def step(self, action):
        # action: 0=straight, 1=left, 2=right
        if action == 1:
            self.direction = (self.direction - 1) % 4
        elif action == 2:
            self.direction = (self.direction + 1) % 4

        new_head = self._next_head(self.direction)

        if self._is_collision(new_head):
            return self.get_state(), -10, True, {}

        self.snake.appendleft(new_head)

        reward = -0.1
        if new_head == self.food:
            self.score += 1
            reward = 10
            self._place_food()
        else:
            self.snake.pop()

        self.steps += 1
        done = self.steps >= self.max_steps
        return self.get_state(), reward, done, {}

    def get_state(self):
        head_x, head_y = self.snake[0]
        food_x, food_y = self.food

        left_dir = (self.direction - 1) % 4
        right_dir = (self.direction + 1) % 4

        danger_straight = int(self._is_collision(self._next_head(self.direction)))
        danger_left = int(self._is_collision(self._next_head(left_dir)))
        danger_right = int(self._is_collision(self._next_head(right_dir)))

        food_left = int(food_x < head_x)
        food_right = int(food_x > head_x)
        food_up = int(food_y < head_y)
        food_down = int(food_y > head_y)

        dir_left = int(self.direction == 2)
        dir_right = int(self.direction == 0)
        dir_up = int(self.direction == 3)
        dir_down = int(self.direction == 1)

        return (
            danger_straight, danger_left, danger_right,
            food_left, food_right, food_up, food_down,
            dir_left, dir_right, dir_up, dir_down,
        )


## 2. Q-Learning Training


In [ ]:
env = SnakeEnv(size=8, max_steps=120)

Q = defaultdict(lambda: np.zeros(3))
alpha = 0.1
gamma = 0.9
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

episodes = 250
scores = []

for _ in range(episodes):
    state = env.reset()
    done = False

    while not done:
        if random.random() < epsilon:
            action = random.randint(0, 2)
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, done, _ = env.step(action)
        Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
        state = next_state

    scores.append(env.score)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)


## 3. Learning Curve


In [ ]:
window = 20
if len(scores) >= window:
    moving_avg = np.convolve(scores, np.ones(window) / window, mode='valid')
else:
    moving_avg = scores

plt.figure(figsize=(8, 4))
plt.plot(scores, label='Score per episode', alpha=0.5)
plt.plot(range(window - 1, window - 1 + len(moving_avg)), moving_avg, label='Moving average')
plt.title('Snake Q-Learning Progress')
plt.xlabel('Episode')
plt.ylabel('Score')
plt.legend()
plt.show()


## 4. Quick Evaluation


In [ ]:
def run_greedy_episode(env, Q):
    state = env.reset()
    done = False
    while not done:
        action = int(np.argmax(Q[state]))
        state, _, done, _ = env.step(action)
    return env.score

scores_eval = [run_greedy_episode(env, Q) for _ in range(5)]
print('Evaluation scores:', scores_eval)
print('Average score:', sum(scores_eval) / len(scores_eval))
